# Crime exploration

Here we will analyze ISTAT and Pagella Politica data prepare it and make it final datasets.

## 0. Setup

### 0.1 Imports and global paths

In [29]:
import io
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


In [30]:
def extract_excel(dir_glob, multitable=True) -> dict[str, pd.DataFrame]:
    tables: dict[str, pd.DataFrame] = {}
    
    file_list = list(dir_glob)
    
    if multitable:

        for file in file_list:
            xlsx_file = pd.ExcelFile(file)
            all_sheets = xlsx_file.sheet_names
            
            for sheet in all_sheets:
                cdf = pd.read_excel(xlsx_file, sheet_name=sheet)
                cdf = cdf.dropna(how='all')
                
                unique_key = f'{Path(file).stem}_{sheet}'.replace(' ', '_').lower()
                
                tables[unique_key] = cdf

    return tables

In [31]:
def extract_csv(dir_glob) -> dict[str, pd.DataFrame]:
    tables: dict[str, pd.DataFrame] = {}
    file_list = list(dir_glob)
    
    for file in file_list:
        with open(raw_data / file, "r", encoding='utf-8-sig') as f:
            lines = f.readlines()
            
            for i, line in enumerate(lines):
                lines[i] = line.replace('"', "'")
                
        df = pd.read_csv(io.StringIO("".join(lines)), quotechar="'", low_memory=False)
        
        unique_key = f'{Path(file).name}'.replace(' ', '_').lower()
        
        tables[unique_key] = df
        
    return tables

In [32]:
root = Path.cwd().parents[0]
raw_data = root / 'data' / 'raw' / 'crime'
processed_data = root / 'data' / 'processed' / 'crime'

crime_csv_files = raw_data.glob('*.csv')
crime_xlsx_files = raw_data.glob('*.xlsx')

crime_xlsx_dfs = extract_excel(crime_xlsx_files, multitable=True)
crime_csv_dfs = extract_csv(crime_csv_files)

## 1 XLSX exploration and cleaning

### 1.1 Exploration
Exploring the structure of the dict we will be working with

In [33]:
print(crime_xlsx_dfs.keys())

print(f"Extracted {len(crime_xlsx_dfs)} tables")
for key, table in crime_xlsx_dfs.items():
    print(f"\n--- {key} ---")
    print(table.shape)
    print(table.head())
    print(table.info())

dict_keys(['tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_1', 'tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_1bis', 'tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_2', 'tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_3', 'tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_4', 'tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_5', 'tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_6', 'tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_7', 'tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_7bis', 'tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_8', 'tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_8bis', 'tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_9', 'tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tav

### 1.2 Keeping only relevant sheets
Filtering out the sheets we do not need, some are categories, others don't have citizenship information.
- **Sheets 9 and 10**: perceived sex and origin of theft/harrassment/pickpocket author.
- **Sheets 7 and 8**: (non-)reporting reasons for individual crimes (pickpocketing, theft, aggression, threat), they do not touch citizenship but can be helpful for general evidences on reporting bias.
- **Sheets 7bis and 8bis**: same as 7 and 8, but for family crimes (car/bike parts theft in domestic environment).

In [34]:
sheets_to_keep = ['7', '8', '9', '10']
for i in range(len(sheets_to_keep)):
    sheets_to_keep[int(i)] = f'tavola_{sheets_to_keep[int(i)]}'
    
print(sheets_to_keep)
clean_xlsx_dfs = {
    k: v for k, v in crime_xlsx_dfs.items() 
    if any(f in k for f in sheets_to_keep) and 'bis' not in k
}

clean_xlsx_dfs.update()

print(list(clean_xlsx_dfs.keys()))

['tavola_7', 'tavola_8', 'tavola_9', 'tavola_10']
['tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_7', 'tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_8', 'tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_9', 'tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_10']


In [35]:
sheet_7 = crime_xlsx_dfs['tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_7']
sheet_8 = crime_xlsx_dfs['tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_8']
sheet_9 = crime_xlsx_dfs['tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_9']
sheet_10 = crime_xlsx_dfs['tavole_reati-contro-la-persona-e-la-proprieta_vittime-ed-eventi_tavola_10']

### 1.3 Sheet 10 exploration and cleaning

In [36]:
print("Sheet 10 details")
display(sheet_10)

Sheet 10 details


,"TAVOLA 10 – Vittime di scippi, rapine, aggressioni per origine italiana o straniera attribuita dalla vittima all'autore del reato, conseguenze motivazioni e tipologia reato. Anni 2022-2023. Valori percentuali",Unnamed: 1,Unnamed: 2,Unnamed: 3
0,TIPOLOGIE DI REATO,Scippo,Rapina,Aggressione
1,ORIGINE AUTORI,NaN,NaN,NaN
2,Di origine italiana,24.5,48,67.4
3,Di origine straniera,44.2,33.9,14.2
4,Di origini italiana e straniera,"2,5*","2,7*",5.4
5,Non so,28.8,"15,4*",13
6,Totale,100,100,100
8,MOTIVAZIONI DELL'ORIGINE AUTORI/AUTRICI,NaN,NaN,NaN
9,"In razione della lingua utilizzata, dall'accento",33,50.3,53.7
10,In ragione dell'apparenza,60.8,37.9,24


In [37]:
column_names_s10 = sheet_10.iloc[0 , 1:4].tolist()

table_10a = sheet_10.iloc[2:6, :4].copy()
table_10a.columns = [c.lower().strip().replace(' ', '_') for c in ['origine_autore'] + column_names_s10]

table_10b = sheet_10.iloc[9:14, :4].copy()
table_10b.columns = [c.lower().strip().replace(' ', '_') for c in ['motivazione_attribuzione'] + column_names_s10]

print(table_10a)
print(table_10b)

                     origine_autore scippo rapina aggressione
2               Di origine italiana   24.5     48        67.4
3              Di origine straniera   44.2   33.9        14.2
4  Di origini italiana e straniera    2,5*   2,7*         5.4
5                            Non so   28.8  15,4*          13
               motivazione_attribuzione scippo rapina aggressione
10            In ragione dell'apparenza   60.8   37.9          24
11    In ragione di un'altra impression   6,2*   5,7*         NaN
12  Il/I ladro/i era/erano conosciuto/i      -      -          22
13                               Non so      -   6,0*        0,4*
14                               Totale    100    100         100


### 1.4 Sheet 9 exploration and cleaning
Has been postponed to Level 2 (dash website) not for claims debunking

In [38]:
print(sheet_9)

   TAVOLA 9 – Vittime di scippi, rapine aggressioni per alcune cratteristiche degli autori e tipologia di reato. Anni 2022-2023  \
0                                  TIPOLOGIE DI REATO                                                                             
1                                  SESSO DEGLI AUTORI                                                                             
2                               Maschi o tutti maschi                                                                             
3                                  Soprattutto maschi                                                                             
4                            Femmiune o tutte femmine                                                                             
5                    Maschi e femmine in ugual numero                                                                             
6                                              non so                              

### 1.5 Sheet 7 exploration and cleaning

In [39]:
column_names_s7 = sheet_7.iloc[0, 1:7].tolist()

table_7 = sheet_7.iloc[1:17, :7].copy()
table_7.columns = [c.lower().strip().replace(' ', '_') for c in ['motivo_non_denuncia'] + column_names_s7]

print(table_7)

                                  motivo_non_denuncia scippo borseggio  \
1   Ha agito per conto suo, se l’è cavata da solo ...   17.3        13   
2     Era un fatto privato, non voleva che si sapesse    NaN       NaN   
3   Non era abbastanza importante, non era abbasta...   12.8      17.5   
4   Non c’erano prove, le forze dell'ordine non po...   23.8      29.4   
5                                 Non era assicurato    3,8*         0   
6   Le forze dell'ordine comunque non avrebbero fa...     20      13.2   
7   Le forze dell'ordine hanno sconsigliato di far...      0         0   
8     Non si voleva perdere tempo a fare la denuncia   11,2*      4,8*   
9                     Timore e paura di rappresaglie       0         0   
10  Non si voleva essere coinvolti in situazioni d...   5,8*      1,4*   
11  La precedente esperienza con la polizia e la g...   4,5*      1,8*   
12  Non è stato rubato nulla/ le cose sono state r...   22.3      33.5   
13                      Le minacce si 

### 1.6 Sheet 8 exploration and cleaning

In [40]:
column_names_s8 = sheet_8.iloc[0, 1:7].tolist()

table_8 = sheet_8.iloc[1:12, :7].copy()
table_8.columns = [c.lower().strip().replace(' ', '_') for c in ['motivo_denuncia'] + column_names_s8]

print(table_8)

                                      motivo_denuncia scippo borseggio  \
1              Per rintracciare il ladro/responsabile   62.3        28   
2           Per impedire al colpevole di farlo ancora   49.6      23.2   
3                    Per ritrovare gli oggetti rubati   53.5      41.8   
4   Per il dovere di informare la polizia o le alt...   15.2      27.8   
5        Per avere il risarcimento dall'assicurazione   5,1*      2,7*   
6   Le forze dell’ordine o le altre autorità compe...      0         0   
7   Perché dovevo denunciare la perdita dei docume...   45.5      44.6   
8   Per avere un maggiore controllo da parte delle...  15,0*      5,3*   
9                           Perché il fatto era grave   5,5*      3,5*   
10                               Per bisogno di aiuto   5,8*      1,0*   
11                       Per bloccare scheda cellular     25      17.6   

   furto_di_oggetti_personali rapina aggressione minaccia  
1                          38   61.7        62.6   

### 1.7 Clean tables values

In [41]:
def clean_istat_value(val):
    if pd.isna(val) or val == '-':
        return np.nan
    val_str = str(val).replace(',', '.').replace('*', '').strip()
    if val_str == '':
        return np.nan
    return float(val_str)

In [42]:
tables_to_clean = [table_7, table_8, table_10a, table_10b]

for table in tables_to_clean:
    numeric_cols = table.columns[1:]
    for col in numeric_cols:
        table[col] = table[col].map(clean_istat_value)
    
    print(f"\n{table.dtypes}")


motivo_non_denuncia               str
scippo                        float64
borseggio                     float64
furto_di_oggetti_personali    float64
rapina                        float64
aggressione                   float64
minaccia                      float64
dtype: object

motivo_denuncia                   str
scippo                        float64
borseggio                     float64
furto_di_oggetti_personali    float64
rapina                        float64
aggressione                   float64
minaccia                      float64
dtype: object

origine_autore        str
scippo            float64
rapina            float64
aggressione       float64
dtype: object

motivazione_attribuzione        str
scippo                      float64
rapina                      float64
aggressione                 float64
dtype: object


### 1.8 Upload excel data to processed

In [43]:
processed_data.mkdir(parents=True, exist_ok=True)

table_7.to_csv(processed_data / "victimization_non_reporting_reasons_2022_2023.csv", index=False)
table_8.to_csv(processed_data / "victimization_reporting_reasons_2022_2023.csv", index=False)
table_10a.to_csv(processed_data / "victimization_perceived_offender_origin_2022_2023.csv", index=False)
table_10b.to_csv(processed_data / "victimization_origin_attribution_reason_2022_2023.csv", index=False)

## 2 CSV exploration and cleaning

In [44]:
denounced_by_citizenship = crime_csv_dfs['sesso,_età,_cittadinanza_(it1,73_230_df_dccv_autvittps_1,1.0).csv']
denounced_by_country = crime_csv_dfs['stranieri_-_paesi_di_cittadinanza_(it1,73_230_df_dccv_autvittps_2,1.0).csv']
convicted_foreign_origin = crime_csv_dfs['reati,_provenienza_geografica_estera_(it1,73_59_df_dccv_condgeo1_2,1.0).csv']
convicted_italian_origin_age = crime_csv_dfs['provenienza_geografica_italiana,_età_(it1,73_59_df_dccv_condgeo1_10,1.0).csv']
convicted_sentence_detail = crime_csv_dfs['dettaglio_reati,_pena_inflitta,_periodo_reclusione_(it1,73_59_df_dccv_condgeo1_8,1.0).csv']

### 2.1 Exploration and upload of `denounced_by_citizenship`

In [45]:
print(denounced_by_citizenship)
print(denounced_by_citizenship['TYPE_CRIME'].nunique(), denounced_by_citizenship['TYPE_CRIME'].unique()[:20])
print(denounced_by_citizenship['CITIZENSHIP'].unique())
print(denounced_by_citizenship['TIME_PERIOD'].min(), denounced_by_citizenship['TIME_PERIOD'].max())

      FREQ Frequenza REF_AREA Territorio DATA_TYPE  \
0        A   Annuale       IT     Italia    OFFEND   
1        A   Annuale       IT     Italia    OFFEND   
2        A   Annuale       IT     Italia    OFFEND   
3        A   Annuale       IT     Italia    OFFEND   
4        A   Annuale       IT     Italia    OFFEND   
...    ...       ...      ...        ...       ...   
41083    A   Annuale       IT     Italia    VICTIM   
41084    A   Annuale       IT     Italia    VICTIM   
41085    A   Annuale       IT     Italia    VICTIM   
41086    A   Annuale       IT     Italia    VICTIM   
41087    A   Annuale       IT     Italia    VICTIM   

                                              Indicatore TYPE_CRIME  \
0      Numero di autori di delitto denunciati/arresta...   MASSMURD   
1      Numero di autori di delitto denunciati/arresta...   MASSMURD   
2      Numero di autori di delitto denunciati/arresta...   MASSMURD   
3      Numero di autori di delitto denunciati/arresta...   MASSMURD

In [46]:
denounced_by_citizenship['TIME_PERIOD'] = denounced_by_citizenship['TIME_PERIOD'].astype(int)
print(denounced_by_citizenship['TIME_PERIOD'].unique)

<bound method Series.unique of 0        2015
1        2016
2        2017
3        2018
4        2019
         ... 
41083    2020
41084    2021
41085    2022
41086    2023
41087    2024
Name: TIME_PERIOD, Length: 41088, dtype: int64>


In [47]:
print(denounced_by_citizenship[['TYPE_CRIME', 'Tipo di delitto']].drop_duplicates().to_string())

       TYPE_CRIME                                                          Tipo di delitto
0        MASSMURD                                                                   Strage
416      INTENHOM                                              Omicidi volontari consumati
896       ROBBHOM                    Omicidi volontari consumati a scopo di furto o rapina
1376     MAFIAHOM                              Omicidi volontari consumati di tipo mafioso
1856    TERRORHOM                         Omicidi volontari consumati a scopo terroristico
2336    ATTEMPHOM                                                          Tentati omicidi
2816    INFANTHOM                                                              Infanticidi
3232      MANSHOM                                               Omicidi preterintenzionali
3712     UNINTHOM                                                          Omicidi colposi
4192      ROADHOM                                    Omicidi colposi da incidente stradale

In [48]:
offenders_denounced_by_citizenship = denounced_by_citizenship[denounced_by_citizenship['DATA_TYPE'] == 'OFFEND']
victims_denounced_by_citizenship = denounced_by_citizenship[denounced_by_citizenship['DATA_TYPE'] == 'VICTIM']
print(offenders_denounced_by_citizenship['DATA_TYPE'].unique())
print(victims_denounced_by_citizenship['DATA_TYPE'].unique())

<StringArray>
['OFFEND']
Length: 1, dtype: str
<StringArray>
['VICTIM']
Length: 1, dtype: str


In [49]:
print(offenders_denounced_by_citizenship.shape)
print(offenders_denounced_by_citizenship.isna().sum())

(25104, 38)
FREQ                                     0
Frequenza                                0
REF_AREA                                 0
Territorio                               0
DATA_TYPE                                0
Indicatore                               0
TYPE_CRIME                               0
Tipo di delitto                          0
SEX                                      0
Sesso                                    0
AGE                                      0
Età                                      0
CITIZENSHIP                              0
Cittadinanza                             0
TIME_PERIOD                              0
Osservazione                             0
OBS_STATUS                           25104
Stato dell'osservazione              25104
NOTE_REF_AREA                        25104
Territorio (NOTE_REF_AREA)           25104
NOTE_DATA_TYPE                           0
Indicatore (NOTE_DATA_TYPE)              0
NOTE_TYPE_CRIME                      21840

In [50]:
offenders = offenders_denounced_by_citizenship.copy()
victims = victims_denounced_by_citizenship.copy()

offenders_empty_cols = offenders.columns[offenders.isna().all()]
victims_empty_cols = victims.columns[offenders.isna().all()]

offenders = offenders.drop(columns=offenders_empty_cols)
victims = victims.drop(columns=victims_empty_cols)

In [51]:
print(offenders.shape, offenders.dtypes)
print(victims.shape, victims.dtypes)
print(offenders['Osservazione'].dtype)

(25104, 20) FREQ                                   str
Frequenza                              str
REF_AREA                               str
Territorio                             str
DATA_TYPE                              str
Indicatore                             str
TYPE_CRIME                             str
Tipo di delitto                        str
SEX                                  int64
Sesso                                  str
AGE                                    str
Età                                    str
CITIZENSHIP                            str
Cittadinanza                           str
TIME_PERIOD                          int64
Osservazione                         int64
NOTE_DATA_TYPE                         str
Indicatore (NOTE_DATA_TYPE)            str
NOTE_TYPE_CRIME                        str
Tipo di delitto (NOTE_TYPE_CRIME)      str
dtype: object
(15984, 20) FREQ                                   str
Frequenza                              str
REF_AREA        

In [52]:
offenders.to_csv(processed_data / "offenders_by_citizenship_2015_2024.csv", index=False)
victims.to_csv(processed_data / "victims_by_citizenship_2015_2024.csv", index=False)

### 2.2 Exploration and uploading of `denounced_by_country`

In [53]:
print(denounced_by_country.columns)

Index(['FREQ', 'Frequenza', 'REF_AREA', 'Territorio', 'DATA_TYPE',
       'Indicatore', 'TYPE_CRIME', 'Tipo di delitto', 'COUNTRY_CITIZEN',
       'Paese di cittadinanza', 'TIME_PERIOD', 'Osservazione', 'OBS_STATUS',
       'Stato dell'osservazione', 'NOTE_REF_AREA',
       'Territorio (NOTE_REF_AREA)', 'NOTE_DATA_TYPE',
       'Indicatore (NOTE_DATA_TYPE)', 'NOTE_TYPE_CRIME',
       'Tipo di delitto (NOTE_TYPE_CRIME)', 'NOTE_COUNTRY_CITIZEN',
       'Paese di cittadinanza (NOTE_COUNTRY_CITIZEN)', 'NOTE_TIME_PERIOD',
       'Tempo (NOTE_TIME_PERIOD)', 'BASE_PER', 'Anno base', 'UNIT_MEAS',
       'Unità di misura', 'UNIT_MULT', 'Unità di moltiplicazione'],
      dtype='str')


In [54]:
print(denounced_by_country.shape)
print(denounced_by_country['DATA_TYPE'].unique())
print(denounced_by_country['COUNTRY_CITIZEN'].nunique(), denounced_by_country['COUNTRY_CITIZEN'].unique()[:20])
print(denounced_by_country['TIME_PERIOD'].min(), denounced_by_country['TIME_PERIOD'].max())

(185120, 30)
<StringArray>
['OFFEND', 'VICTIM']
Length: 2, dtype: str
210 <StringArray>
['WORLD',    'AL',    'AD',    'AT',    'BE',    'BY',    'BA',    'BG',
    'CZ',   'X62',    'CY',    'HR',    'DK',    'EE',    'FI',    'FR',
    'MQ',    'DE',    'GR',    'IE']
Length: 20, dtype: str
2015 2024


In [55]:
# rimozione codice paese X62: si riferisce alla cecoslovacchia
denounced_by_country = denounced_by_country[denounced_by_country['COUNTRY_CITIZEN'] != 'X62']

# Split Offenders and Victims
offenders_by_country = denounced_by_country[denounced_by_country['DATA_TYPE'] != 'OFFEND'].copy()
victims_by_country = denounced_by_country[denounced_by_country['DATA_TYPE'] != 'VICTIM'].copy()

# Detect and remove empty cols
offenders_by_country_empty_cols = offenders_by_country.columns[offenders_by_country.isna().all()]
victims_by_country_empty_cols = victims_by_country.columns[victims_by_country.isna().all()]

offenders_by_country = offenders_by_country.drop(columns=offenders_by_country_empty_cols)
victims_by_country = victims_by_country.drop(columns=victims_by_country_empty_cols)

In [56]:
print(offenders_by_country.shape, offenders_by_country.dtypes)
print(victims_by_country.shape, victims_by_country.dtypes)
print(offenders_by_country['Osservazione'].dtype)
print(victims_by_country['Osservazione'].dtype)

(69403, 18) FREQ                                              str
Frequenza                                         str
REF_AREA                                          str
Territorio                                        str
DATA_TYPE                                         str
Indicatore                                        str
TYPE_CRIME                                        str
Tipo di delitto                                   str
COUNTRY_CITIZEN                                   str
Paese di cittadinanza                             str
TIME_PERIOD                                     int64
Osservazione                                    int64
NOTE_DATA_TYPE                                    str
Indicatore (NOTE_DATA_TYPE)                       str
NOTE_TYPE_CRIME                                   str
Tipo di delitto (NOTE_TYPE_CRIME)                 str
NOTE_COUNTRY_CITIZEN                              str
Paese di cittadinanza (NOTE_COUNTRY_CITIZEN)      str
dtype: object
(1

In [57]:
# DFs uploads as CSVs
offenders_by_country.to_csv(processed_data / "offenders_by_country_2015_2024.csv", index=False)
victims_by_country.to_csv(processed_data / "victims_by_country_2015_2024.csv", index=False)